In [ ]:
import subprocess, json
from dialoghelper import find_msgs

async def _get_context(limit: int = None) -> str:
    """Fetch dialog messages as formatted context."""
    recent = await find_msgs(limit=limit)
    lines = []
    for m in recent:
        role = 'User' if m['msg_type'] == 'prompt' else 'Code' if m['msg_type'] == 'code' else 'Note'
        content = m.get('content', '')[:300]
        lines.append(f"{role}: {content}")
    return "\n\n".join(lines)

async def pi_ask(prompt: str, include_context: bool = False, context_limit: int = None, timeout: int = 300) -> str:
    """Ask pi coding agent and return the response.
    
    Args:
        prompt: The question for pi
        include_context: Whether to include recent dialog messages
        context_limit: Number of recent messages to include (None=all)
        timeout: Seconds to wait
    
    Returns:
        pi's text response
    """
    full_prompt = prompt
    if include_context:
        context = await _get_context(context_limit)
        full_prompt = f"Previous conversation context:\n{context}\n\n---\n\nNew question: {prompt}"
    
    cmd = ['pi', '-p', '--mode', 'json', full_prompt]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    
    for line in result.stdout.strip().split('\n'):
        try:
            data = json.loads(line)
            if data.get('type') == 'agent_end':
                for msg in data.get('messages', []):
                    if msg.get('role') == 'assistant':
                        text_parts = [p.get('text', '') for p in msg.get('content', []) if p.get('type') == 'text']
                        return ''.join(text_parts)
        except json.JSONDecodeError:
            continue
    return ""

In [ ]:
async def claude_ask(prompt: str, include_context: bool = False, context_limit: int = None, timeout: int = 300) -> str:
    """Ask claude code agent and return the response.
    
    Args:
        prompt: The question for claude
        include_context: Whether to include recent dialog messages
        context_limit: Number of recent messages to include (None=all)
        timeout: Seconds to wait
    
    Returns:
        claude's text response
    """
    full_prompt = prompt
    if include_context:
        context = await _get_context(context_limit)
        full_prompt = f"Previous conversation context:\n{context}\n\n---\n\nNew question: {prompt}"
    
    cmd = ['claude', full_prompt]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return result.stdout.strip()

&`pi_ask`

&`claude_ask`